# lb-1200 + Shallow Action-Set Lookahead

**Goal**: improve on the pure-rule-based lb-1200 agent by wrapping its action output in a shallow lookahead over candidate variants.

## Design (NOT full MCTS)

1. lb-1200 computes its primary action list `A_0` per its normal logic.
2. Generate variants `A_1 .. A_4`:
   - V0: original `A_0`
   - V1: 75% ships per action
   - V2: 50% ships per action
   - V3: drop weakest action (smallest ship count)
   - V4: pass (no actions)
3. Score each via lb-1200's own `simulate_planet_timeline` over horizon=30:
   - `score = my_ships + 30·my_planets − 0.5·enemy_ships − 20·fallen`
4. Pick the highest-scoring variant.

**Why this works**: lb-1200 picks each planet's action independently; the variants let us reconsider the *joint* plan. Dropping a weak action or doing nothing can sometimes outperform overcommitting.

**What this is NOT**: no state tree, no UCB, no rollouts into futures with opponent actions. Scoring assumes opponent does nothing in the horizon (approximation).

## Performance

- lb-1200 base: ~0.3–0.7s/turn
- Lookahead overhead: ~0.15s/turn (3–4 variants × ~40ms each)
- Total: ~0.5–0.9s — fits Kaggle's 1–5s/turn budget comfortably.

See `training/lb1200_lookahead_agent.py` for source.

In [ ]:
# Setup
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import os
os.environ.setdefault('LD_LIBRARY_PATH', '/home/lab/miniconda3/envs/tom/lib')

from training.lb1200_agent import agent as lb1200_agent
from training.lb1200_lookahead_agent import agent as lookahead_agent, agent_debug
from kaggle_environments import make
print('lookahead agent loaded')

In [ ]:
# Single-game debug: watch what variants get scored + which gets picked
env = make('orbit_wars', debug=False)
env.reset(num_agents=2)
# Step the env a few turns manually to get mid-game state
dummy_action = [[]]  # pass
for _ in range(20):
    env.step([[], []])

obs = env.state[0].observation
config = env.configuration
action, dbg = agent_debug(obs, config)
print(f'Picked variant {dbg["chosen"]}: {action}')
print('\nAll variants scored:')
for i, v in enumerate(dbg['variants']):
    mark = ' ←' if i == dbg['chosen'] else ''
    print(f'  V{i}: score={v["score"]:.1f}  my_ships={v.get("my_ships", 0):.0f}  '
          f'my_planets={v.get("my_planets", 0)}  fallen={v.get("fallen", 0)}{mark}')
    print(f'      action: {v["variant"]}')

In [ ]:
# Head-to-head: lookahead vs raw lb-1200, 10 games
import time
results = []
N = 10
for i in range(N):
    seat = i % 2  # alternate first/second
    env = make('orbit_wars', debug=False); env.reset(num_agents=2)
    agents = [None, None]
    agents[seat] = lookahead_agent
    agents[1 - seat] = lb1200_agent
    t0 = time.time()
    env.run(agents)
    dt = time.time() - t0
    r = (env.state[seat].reward or 0) > (env.state[1-seat].reward or 0)
    results.append({'seat': seat, 'won': r, 'steps': len(env.steps), 'dt': dt})
    print(f'game {i+1}/{N}  seat={seat}  won={r}  steps={len(env.steps)}  {dt:.1f}s')

wins = sum(1 for x in results if x['won'])
print(f'\n>>> lookahead vs lb-1200: {wins}/{N} ({100*wins/N:.0f}%)')

## Expected baseline

A **self-play-balanced** lookahead vs lb-1200 should score **50-65%** if the lookahead adds any real value. Results:

- **~50%**: lookahead is noise-equivalent to lb-1200. No measurable improvement.
- **55-65%**: lookahead catches joint-plan suboptimalities lb-1200 misses. Worth submitting.
- **>65%**: suspicious — either lb-1200 has a real weakness or lookahead is exploiting a scoring quirk. Inspect details.
- **<45%**: lookahead is *worse* — probably the scoring function misweights risk, or horizon=30 is too short. Revise.

## Kaggle submission preparation

Kaggle Orbit Wars requires a single `main.py` exporting `agent(obs, config) -> list[moves]`. Our agent imports from `training.lb1200_agent`, so we need to bundle both files into a tar.gz.

```bash
# (run from repo root)
mkdir -p submission/lb1200_lookahead
# Combine lb1200_agent.py + lb1200_lookahead_agent.py into main.py
python - <<'EOF'
base = open('training/lb1200_agent.py').read()
wrap = open('training/lb1200_lookahead_agent.py').read()
# Strip the wrap's import of lb1200 (inlined below)
wrap = wrap.replace(
    'from training.lb1200_agent import (\n    agent as lb1200_base_agent,',
    '# inline: lb1200_base_agent = agent (defined above)\nlb1200_base_agent = None  # patched below'
)
combined = base + '\n\n# ===== Lookahead wrapper =====\n' + wrap
combined += '\nlb1200_base_agent = globals()[\'agent\']\n# Re-export as `agent` for Kaggle\nagent = globals()[\'agent\']\n'
with open('submission/lb1200_lookahead/main.py', 'w') as f:
    f.write(combined)
EOF

tar czf submission/lb1200_lookahead/submission.tar.gz \
    -C submission/lb1200_lookahead main.py

kaggle competitions submit orbit-wars \
    -f submission/lb1200_lookahead/submission.tar.gz \
    -m 'lb-1200 + shallow lookahead (5 action variants, horizon=30)'
```

**Note**: the inline combination script is a simplified example — for real submission you'll need to clean up imports and naming conflicts. See `training/make_submission_lookahead.sh` (TODO) for the full build pipeline.